# 08 · Credit Derivatives: CDS & Indices

Single-name CDS, indices, options, and tranches share hazard-curve inputs. This notebook wires the necessary market data and showcases each instrument.

### Learning Objectives
- Construct discount/hazard/base-correlation data for credit pricing
- Price a single-name CDS with par spread / PV01 metrics
- Value a CDS index and an option on single-name protection
- Evaluate mezzanine tranche PV and spreads off base-correlation curves

In [1]:
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import (
    DiscountCurve,
    HazardCurve,
    BaseCorrelationCurve,
    CreditIndexData,
)
from finstack.core.market_data.surfaces import VolSurface
from finstack.valuations.instruments import (
    CreditDefaultSwap,
    CDSIndex,
    CDSOption,
    CDSTranche,
)
from finstack.valuations.pricer import standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9980), (1.0, 0.9960), (3.0, 0.9850), (5.0, 0.9600)],
    )
)
acme_curve = HazardCurve(
    "ACME-HZD",
    as_of,
    [(0.0, 0.0120), (3.0, 0.0180), (5.0, 0.0220)],
    recovery_rate=0.40,
)
index_curve = HazardCurve(
    "CDX-IG-HZD",
    as_of,
    [(0.0, 0.0100), (3.0, 0.0160), (5.0, 0.0190), (7.0, 0.0210)],
    recovery_rate=0.40,
)
market.insert_hazard(acme_curve)
market.insert_hazard(index_curve)

base_corr = BaseCorrelationCurve(
    "CDX-IG-BC",
    [(0.03, 0.10), (0.06, 0.12), (0.10, 0.15), (0.30, 0.20), (0.70, 0.23), (1.00, 0.25)],
)
index_data = CreditIndexData(125, 0.40, index_curve, base_corr)
market.insert_base_correlation(base_corr)
market.insert_credit_index("CDX.NA.IG", index_data)

market.insert_surface(
    VolSurface(
        "CDS-VOL",
        expiries=[0.5, 1.0, 3.0, 5.0],
        strikes=[0.0100, 0.0200, 0.0400],
        grid=[
            [0.45, 0.40, 0.35],
            [0.42, 0.38, 0.33],
            [0.38, 0.35, 0.30],
            [0.35, 0.32, 0.28],
        ],
    )
)

registry = standard_registry()
notional = Money(10_000_000, USD)
start = as_of + timedelta(days=1)
maturity = date(2029, 1, 2)


## 1. Single-Name CDS
`CreditDefaultSwap.buy_protection` builds running-spread CDS with hazard and discount curves. Request par spread / PV01.

In [2]:
cds = CreditDefaultSwap.buy_protection(
    "ACME-CDS",
    notional,
    spread_bp=120.0,
    start_date=start,
    maturity=maturity,
    discount_curve="USD-OIS",
    credit_curve="ACME-HZD",
)
cds_res = registry.price_with_metrics(cds, "discounting", market, as_of=as_of, metrics=["par_spread", "pv01"])
print("CDS PV:", cds_res.value)
print("Par spread:", cds_res.measures.get("par_spread"))
print("PV01:", cds_res.measures.get("pv01"))


TypeError: PricerRegistry.price_with_metrics() got multiple values for argument 'as_of'

## 2. CDS Index
Indices reference `CreditIndexData` containing hazard and base-correlation information. Use fixed coupons (100 bp typical) and request par spreads.

In [3]:
index = (
    CDSIndex.builder("CDX-TRAD")
    .index_name("CDX.NA.IG")
    .series(42)
    .version(1)
    .notional(Money(25_000_000, USD))
    .fixed_coupon_bp(100.0)
    .start_date(start)
    .maturity(maturity)
    .discount_curve("USD-OIS")
    .credit_curve("CDX-IG-HZD")
    .build()
)
index_res = registry.price_with_metrics(index, "discounting", market, as_of=as_of, metrics=["par_spread"])
print("CDX index PV:", index_res.value)
print("Par spread:", index_res.measures.get("par_spread"))


TypeError: PricerRegistry.price_with_metrics() got multiple values for argument 'as_of'

## 3. CDS Options
Options reference the same hazard curve and discount curve plus a vol surface keyed by strike in spread basis points.

In [ ]:
cds_option = (
    CDSOption.builder("ACME-CDSOPT")
    .notional(Money(5_000_000, USD))
    .strike(0.015)
    .expiry(date(2025, 1, 2))
    .cds_maturity(maturity)
    .discount_curve("USD-OIS")
    .credit_curve("ACME-HZD")
    .vol_surface("CDS-VOL")
    .option_type("call")
    .build()
)
opt_res = registry.price_with_metrics(cds_option, "discounting", market, as_of=as_of, metrics=["vega"])
print("CDS option PV:", opt_res.value)
print("CDS option vega:", opt_res.measures.get("vega"))


## 4. Mezzanine Tranche
Tranches reference the credit index base correlation curve. Provide attachment/detachment levels, running coupon, and payment frequency.

In [4]:
tranche = (
    CDSTranche.builder("CDX-MEZ")
    .index_name("CDX.NA.IG")
    .series(42)
    .attach_pct(3.0)
    .detach_pct(7.0)
    .notional(Money(10_000_000, USD))
    .maturity(maturity)
    .running_coupon_bp(500.0)
    .discount_curve("USD-OIS")
    .credit_index_curve("CDX.NA.IG")
    .side("buy_protection")
    .payments_per_year(4)
    .build()
)
tranche_res = registry.price_with_metrics(tranche, "discounting", market, as_of=as_of, metrics=["par_spread", "expected_loss"])
print("Tranche PV:", tranche_res.value)
print("Par spread:", tranche_res.measures.get("par_spread"))
print("Expected loss:", tranche_res.measures.get("expected_loss"))


TypeError: PricerRegistry.price_with_metrics() got multiple values for argument 'as_of'

## Summary
- Hazard curves + discount curves are sufficient for single-name CDS; indices add base-correlation via `CreditIndexData`.
- CDS options require consistent vol surfaces expressed in spread space.
- Tranches consume the same index object, ensuring base-correlation, hazard, and premium conventions remain aligned.